In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/burnout_dataset.csv")

print("Dataset loaded successfully")
print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df.head(3))

Dataset loaded successfully
Shape  : (1600, 19)
Columns: ['employee_id', 'week_number', 'burnout_label', 'tasks_planned', 'tasks_completed', 'completion_rate', 'rework_count', 'focus_sessions', 'avg_focus_duration', 'context_switches', 'idle_time_mins', 'work_hours_per_day', 'after_hours_logins', 'breaks_taken', 'weekend_work_days', 'error_rate', 'submission_delays', 'revision_requests', 'mental_fatigue_score']

First 3 rows:
  employee_id  week_number burnout_label  tasks_planned  tasks_completed  \
0        E001            1           Low             15               13   
1        E001            2           Low             16               15   
2        E001            3           Low             16               14   

   completion_rate  rework_count  focus_sessions  avg_focus_duration  \
0            0.906             0               4                71.0   
1            0.948             0               4               118.2   
2            0.894             0               6 

In [7]:
df = df.sort_values(
    ["employee_id", "week_number"]
).reset_index(drop=True)

feature_cols = [
    "completion_rate",
    "rework_count",
    "focus_sessions",
    "avg_focus_duration",
    "context_switches",
    "idle_time_mins",
    "work_hours_per_day",
    "after_hours_logins",
    "breaks_taken",
    "weekend_work_days",
    "error_rate",
    "submission_delays",
    "revision_requests",
    "mental_fatigue_score"
]

print(f"Data sorted by employee and week")
print(f"Feature columns : {len(feature_cols)}")
print(f"\nSample — E001 weeks 1 to 3:")
print(df[df["employee_id"] == "E001"][
    ["employee_id", "week_number", "burnout_label"]
    + feature_cols[:4]
].head(3))

Data sorted by employee and week
Feature columns : 14

Sample — E001 weeks 1 to 3:
  employee_id  week_number burnout_label  completion_rate  rework_count  \
0        E001            1           Low            0.906             0   
1        E001            2           Low            0.948             0   
2        E001            3           Low            0.894             0   

   focus_sessions  avg_focus_duration  
0               4                71.0  
1               4               118.2  
2               6                84.0  


In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Features normalized to range [0, 1]")
print(f"\nSample after normalization — E001 weeks 1 to 3:")
print(df[df["employee_id"] == "E001"][
    ["employee_id", "week_number", "burnout_label"]
    + feature_cols[:4]
].head(3).round(3))

print(f"\nMin values:")
print(df[feature_cols].min().round(3))
print(f"\nMax values:")
print(df[feature_cols].max().round(3))

Features normalized to range [0, 1]

Sample after normalization — E001 weeks 1 to 3:
  employee_id  week_number burnout_label  completion_rate  rework_count  \
0        E001            1           Low            0.867           0.0   
1        E001            2           Low            0.927           0.0   
2        E001            3           Low            0.849           0.0   

   focus_sessions  avg_focus_duration  
0             0.6               0.555  
1             0.6               0.985  
2             1.0               0.674  

Min values:
completion_rate         0.0
rework_count            0.0
focus_sessions          0.0
avg_focus_duration      0.0
context_switches        0.0
idle_time_mins          0.0
work_hours_per_day      0.0
after_hours_logins      0.0
breaks_taken            0.0
weekend_work_days       0.0
error_rate              0.0
submission_delays       0.0
revision_requests       0.0
mental_fatigue_score    0.0
dtype: float64

Max values:
completion_rate      

In [10]:
df_drift = df.copy()

for col in feature_cols:
    df_drift[f"{col}_baseline"] = (
        df_drift.groupby("employee_id")[col]
        .transform(lambda x: x.rolling(
            window=4, min_periods=1
        ).mean().shift(1))
    )

    df_drift[f"{col}_std"] = (
        df_drift.groupby("employee_id")[col]
        .transform(lambda x: x.rolling(
            window=4, min_periods=1
        ).std().shift(1))
        .fillna(0.01)
        .replace(0, 0.01)
    )

    df_drift[f"{col}_drift"] = (
        (df_drift[col] - df_drift[f"{col}_baseline"]) /
        df_drift[f"{col}_std"]
    )

drift_cols = [f"{col}_drift" for col in feature_cols]
df_drift[drift_cols] = (
    df_drift[drift_cols]
    .replace([float("inf"), float("-inf")], 0)
    .fillna(0)
)
df_drift[drift_cols] = df_drift[drift_cols].clip(-3, 3)
df_drift["overall_drift_score"] = (
    df_drift[drift_cols].abs().mean(axis=1)
)

print("Drift scores computed")
print(f"New columns added : {len(drift_cols)} drift scores")
print(f"Total columns now : {df_drift.shape[1]}")
print(f"\nOverall drift score by burnout label:")
print(df_drift.groupby("burnout_label")[
    "overall_drift_score"
].mean().round(3))

Drift scores computed
New columns added : 14 drift scores
Total columns now : 62

Overall drift score by burnout label:
burnout_label
High      1.306
Low       1.000
Medium    1.354
Name: overall_drift_score, dtype: float64


In [11]:
print("Average drift score per week:")
print(df_drift.groupby("week_number")[
    "overall_drift_score"
].mean().round(3).to_string())

print("\nAverage drift score by label and week range:")
df_drift["week_range"] = pd.cut(
    df_drift["week_number"],
    bins=[0, 6, 11, 16],
    labels=["Weeks 1-6 (Low)",
            "Weeks 7-11 (Medium)",
            "Weeks 12-16 (High)"]
)
print(df_drift.groupby("week_range")[
    "overall_drift_score"
].mean().round(3))

Average drift score per week:
week_number
1     0.000
2     1.912
3     1.154
4     1.007
5     0.989
6     0.935
7     2.661
8     1.417
9     0.916
10    0.719
11    1.059
12    2.399
13    1.414
14    0.965
15    0.732
16    1.019

Average drift score by label and week range:
week_range
Weeks 1-6 (Low)        1.000
Weeks 7-11 (Medium)    1.354
Weeks 12-16 (High)     1.306
Name: overall_drift_score, dtype: float64


C:\Users\Shubham\AppData\Local\Temp\ipykernel_18504\4008618952.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_drift.groupby("week_range")[


In [12]:
from sklearn.preprocessing import LabelEncoder
import os

le = LabelEncoder()
df_drift["label_encoded"] = le.fit_transform(
    df_drift["burnout_label"]
)

print("Labels encoded")
print(f"Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"\nLabel distribution:")
print(df_drift["burnout_label"].value_counts())

os.makedirs("../data/processed", exist_ok=True)
df_drift.to_csv(
    "../data/processed/burnout_with_drift.csv",
    index=False
)

saved = pd.read_csv(
    "../data/processed/burnout_with_drift.csv"
)
print(f"\n Processed dataset saved")
print(f"File : data/processed/burnout_with_drift.csv")
print(f"Shape: {saved.shape}")

Labels encoded
Mapping: {'High': 0, 'Low': 1, 'Medium': 2}

Label distribution:
burnout_label
Low       600
Medium    500
High      500
Name: count, dtype: int64

 Processed dataset saved
File : data/processed/burnout_with_drift.csv
Shape: (1600, 64)
